# 2026.11.23 TIL

In [156]:
import pandas as pd
import numpy as np
import seaborn as sns

print("pandas", pd.__version__, "| numpy", np.__version__, "| seaborn", sns.__version__)

pandas 3.1.1 | numpy 2.5.5 | seaborn 0.13.2


# 표를 열고 흝어보기
- 목표 타이타닉 승객 표를 불러와 크기 자료형 빈칸 개수 숫자 요약을 읽는다.

## 데이터 불러오기
- seaborn 에 들어 있는 타이타닉 데이터를 df 로 불러오고 앞 5불을 보세요.

In [157]:
df = sns.load_dataset("titanic")
df.head()

,survived,pclass,sex,age,sibsp,parch,fare,embarked,class,who,adult_male,deck,embark_town,alive,alone
0,0,3,male,22.0,1,0,7.2500,S,Third,man,True,NaN,Southampton,no,False
1,1,1,female,38.0,1,0,71.2833,C,First,woman,False,C,Cherbourg,yes,False
2,1,3,female,26.0,0,0,7.9250,S,Third,woman,False,NaN,Southampton,yes,True
3,1,1,female,35.0,1,0,53.1000,S,First,woman,False,C,Southampton,yes,False
4,0,3,male,35.0,0,0,8.0500,S,Third,man,True,NaN,Southampton,no,True


## 몇 행 몇 열?
- 표의 크기를 확인하세요.

In [158]:
df.shape

(891, 15)

## 자료형과 빈칸 한눈에
- info() 로 칸마다 값이 몇 개 있는지, 자료형이 무엇인지 보세요.

In [159]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 15 columns):
 #   Column       Non-Null Count  Dtype   
---  ------       --------------  -----   
 0   survived     891 non-null    int64   
 1   pclass       891 non-null    int64   
 2   sex          891 non-null    str     
 3   age          714 non-null    float64 
 4   sibsp        891 non-null    int64   
 5   parch        891 non-null    int64   
 6   fare         891 non-null    float64 
 7   embarked     889 non-null    str     
 8   class        891 non-null    category
 9   who          891 non-null    str     
 10  adult_male   891 non-null    bool    
 11  deck         203 non-null    category
 12  embark_town  889 non-null    str     
 13  alive        891 non-null    str     
 14  alone        891 non-null    bool    
dtypes: bool(2), category(2), float64(2), int64(4), str(5)
memory usage: 80.7 KB


## 숫자 칸 요약
- describe()로 숫자 칸의 평균 사분위수 최댓값을 보세요.

In [160]:
df.describe()

,survived,pclass,age,sibsp,parch,fare
count,891.000000,891.000000,714.000000,891.000000,891.000000,891.000000
mean,0.383838,2.308642,29.699118,0.523008,0.381594,32.204208
std,0.486592,0.836071,14.526497,1.102743,0.806057,49.693429
min,0.000000,1.000000,0.420000,0.000000,0.000000,0.000000
25%,0.000000,2.000000,20.125000,0.000000,0.000000,7.910400
50%,0.000000,3.000000,28.000000,0.000000,0.000000,14.454200
75%,1.000000,3.000000,38.000000,1.000000,0.000000,31.000000
max,1.000000,3.000000,80.000000,8.000000,6.000000,512.329200


## 글자로 착각한 숫자 고치기
- 목표 ￦ 기호와 쉽표가 붙은 가격이 왜 계산되지 않는지 확인하고, 숫자로 바꿔 평균을 구한다.

In [161]:
shop = pd.DataFrame({
    "상품": ["텀블러", "에코백", "무선 이어폰", "충전기", "노트", "보조배터리"],
    "가격": ["₩12,000", "₩8,500", "₩129,000", "₩19,900", "₩3,000", "₩35,000"],
})
shop.dtypes

상품    str
가격    str
dtype: object

## 평균을 구해 보면?
- 가격의 평균을 구해 보세요.

In [162]:
step1 = shop["가격"].str.replace("₩","").str.replace(",","")
print(step1.tolist())
print(step1.dtypes)

['12000', '8500', '129000', '19900', '3000', '35000']
str


## 이번에는 기호를 지운 뒤 pd.to_numeric 으로 숫자로 바꾸고, 자료형과 평균을 다시 확인하세요.

In [163]:

shop["가격"] = shop["가격"].astype(str).str.replace("₩","").str.replace(",","")
shop["가격"] = pd.to_numeric(shop["가격"])
print(shop.dtypes)
print("평균 가격:", round(shop["가격"].mean()))


상품      str
가격    int64
dtype: object
평균 가격: 34567


## 빈칸 채우기 (나이, 항구, 갑판)
- 목표 칸의 성격(숫자/범주/대부분 빈칸)에 따라 다른 방법으로 빈칸을 채우고, 그 선택이 분포를 바꾼더는 것을 확인한다.

> 왜 빈칸부터 채울까요? <br> 머신러닝 모델은 표에 빈칸(NaN)이나 글자가 남아 있으면 학습을 거부하거나, 거부하지 않더라도 엉뚱한 값으로 배웁니다.<br> 그래서 모델에 넣기 전에 빈칸 자료형 이상치를 정리하는 이 작업을 데이터 전처리라고 하며, 이어지는 모든 장의 출발접입니다.

## 빈칸 개수 세기
- 칸마다 빈칸이 몇 개인지 세어보세요.

In [164]:
df.isnull().sum()

survived         0
pclass           0
sex              0
age            177
sibsp            0
parch            0
fare             0
embarked         2
class            0
who              0
adult_male       0
deck           688
embark_town      2
alive            0
alone            0
dtype: int64

## 나이:평균? 중앙값?
- 나이의 평균과 중앙값을 비교한 뒤, 빈칸을 평균으로 채우세요.
   

In [165]:
# 나이가 비어 있는 승객이 몇 명이고, 어떤 사람들인지 먼저 보세요.

empty_age = df["age"].isnull()
print("나이가 비어있는 승객 :",empty_age.sum(),"명")

# 평균과 중앙값을 비교한 뒤 평균으로 채우고, 위에서 본 승객들의 나이가 어떻게 바뀌었는지 확인하세요.
print("평균:",round(df["age"].mean()),"::","중앙값:",df["age"].median())
age_mean = round(df["age"].mean())
df.loc[:,"age"] = df["age"].fillna(age_mean)
df.loc[empty_age,["sex","age","class"]].head()


나이가 비어있는 승객 : 177 명
평균: 30 :: 중앙값: 28.0


,sex,age,class
5,male,30.0,Third
17,male,30.0,Second
19,female,30.0,Third
26,male,30.0,Third
28,female,30.0,Third


## 항구: 최빈값
- 탑승 항구(embar_town)는 글자라 평균이 없습니다. 가장 많이 탄 항구로 채우세요.


In [174]:
# 항구별 승객 수를 빈칸(NaN)까지 포함해 세어 보세요.
(df["embark_town"].value_counts(dropna=False) )


# 최빈값을 구해 빈칸을 채우고, 다시 세어 NaN이 사라졌는지 확인하세요.
mode_town = df["embark_town"].mode()[0]
df.loc[:,"embark_town"] = df["embark_town"].fillna(mode_town)
df["embark_town"].value_counts(dropna=False)



embark_town
Southampton    646
Cherbourg      168
Queenstown      77
Name: count, dtype: int64

## embarked 를 town 에 맞추기
- embarked(S/C/Q)는 embark_town의 첫 글자입니다. town 을 보고 다시 만드세요.

In [182]:
str1 = df["embark_town"].str[0]    
df.loc[:,"embarked"] = str1
df[["embarked","embark_town"]].isnull().sum()

embarked       0
embark_town    0
dtype: int64

## 갑판: 앞사람 값으로 채우면?
- 수업 교안처럼 갑판(deck) 빈칸을 앞 뒤 승객 값으로 채운 사본을 만들어, 갑판별 인원을 원본과 비교하세요.

In [188]:
test = df["deck"].ffill().bfill()
pd.DataFrame({"원본":df["deck"].value_counts(),"ffill bill 후": test.value_counts()})


,원본,ffill bill 후
deck,,
A,15,77
B,47,201
C,59,260
D,33,126
E,32,148
F,13,55
G,4,24


## 갑판: '모름'으로 남기기
- deck 이 어떤 값을 받을 수 있는지 먼저 확인한 뒤, 'unknown' 범주를 추가하고 그것으로 빈칸을 채웁니다.

In [203]:
# deck 의 자료형과, 받을 수 있는 값의 목록(범주)을 확인하세요.
print(df["deck"].dtypes)
df["deck"].cat.categories

# dock 에 'Unknown' 범주를 추가하고 그것으로 빈칸을 채운 뒤, 범주 목록이 어떻게 바뀌었는지 다시 확인하세요.
#df["deck"] = df["deck"].cat.add_categories(["Unknown"]).fillna('Unknown')
print(df["deck"].cat.categories.tolist())
print(df["deck"].value_counts())
df.isnull().sum().sum()



category
['A', 'B', 'C', 'D', 'E', 'F', 'G', 'Unknown']
deck
Unknown    688
C           59
B           47
D           33
E           32
A           15
F           13
G            4
Name: count, dtype: int64


np.int64(0)